In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/khmer-similarity-char/khmer_semantic_similarity_dataset_chars.csv


In [2]:
# ===================================================
# Khmer Semantic Similarity Detection (RNN)
# ===================================================

import pandas as pd
from collections import Counter
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dot
from tensorflow.keras.models import Model
import tensorflow as tf

2025-12-21 12:47:08.496979: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766321228.677182      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766321228.729521      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766321229.186632      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766321229.186673      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766321229.186676      24 computation_placer.cc:177] computation placer alr

In [3]:
# ----------------------------
# Step 1: Load CSV
# ----------------------------
# First row contains headers
df = pd.read_csv("/kaggle/input/khmer-similarity-char/khmer_semantic_similarity_dataset_chars.csv", header=0)
df.columns

Index(['word1', 'word2', 'similarity', 'chars1', 'chars2'], dtype='object')

In [4]:
df.head()

,word1,word2,similarity,chars1,chars2
0,កន្លែង,ទី,1,"['ក', 'ន', '្', 'ល', 'ែ', 'ង']","['ទ', 'ី']"
1,កន្លែង,តំបន់,1,"['ក', 'ន', '្', 'ល', 'ែ', 'ង']","['ត', 'ំ', 'ប', 'ន', '់']"
2,កន្លែង,ស្ថាន,1,"['ក', 'ន', '្', 'ល', 'ែ', 'ង']","['ស', '្', 'ថ', 'ា', 'ន']"
3,ទី,តំបន់,1,"['ទ', 'ី']","['ត', 'ំ', 'ប', 'ន', '់']"
4,ទី,ស្ថាន,1,"['ទ', 'ី']","['ស', '្', 'ថ', 'ា', 'ន']"


In [5]:
# ----------------------------
# Step 2: Build character vocabulary
# ----------------------------

all_chars = [c for word in df['chars1'] for c in word] + \
            [c for word in df['chars2'] for c in word]

char_counts = Counter(all_chars)
char2idx = {c: i+1 for i, c in enumerate(char_counts)}  # 0 reserved for padding
vocab_size = len(char2idx) + 1


In [6]:
vocab_size

78

In [7]:
# Encode characters to integers
def encode(chars):
    return [char2idx[c] for c in chars]

df['seq1'] = df['chars1'].apply(encode)
df['seq2'] = df['chars2'].apply(encode)

# 🔧 DEFINE max_len HERE
max_len = max(
    df['seq1'].apply(len).max(),
    df['seq2'].apply(len).max()
)

# ==================== SAVE PREPROCESSING DATA FOR TESTING ====================
import pickle

print("="*80)
print("SAVING PREPROCESSING DATA")
print("="*80)

# Save char2idx
with open('char2idx.pkl', 'wb') as f:
    pickle.dump(char2idx, f)
print(f"✓ Saved char2idx.pkl ({len(char2idx)} characters)")

# Save max_len
with open('max_len.pkl', 'wb') as f:
    pickle.dump(max_len, f)
print(f"✓ Saved max_len.pkl (max_len={max_len})")

# Optional: Save idx2char
try:
    with open('idx2char.pkl', 'wb') as f:
        pickle.dump(idx2char, f)
    print("✓ Saved idx2char.pkl")
except NameError:
    print("  (idx2char not defined, skipping)")

print("✓ All preprocessing data saved successfully!")
print("="*80 + "\n")


SAVING PREPROCESSING DATA
✓ Saved char2idx.pkl (77 characters)
✓ Saved max_len.pkl (max_len=100)
  (idx2char not defined, skipping)
✓ All preprocessing data saved successfully!



In [8]:
df.head()

,word1,word2,similarity,chars1,chars2,seq1,seq2
0,កន្លែង,ទី,1,"['ក', 'ន', '្', 'ល', 'ែ', 'ង']","['ទ', 'ី']","[1, 2, 3, 2, 4, 5, 2, 6, 2, 4, 5, 2, 7, 2, 4, ...","[1, 2, 12, 2, 4, 5, 2, 13, 2, 11]"
1,កន្លែង,តំបន់,1,"['ក', 'ន', '្', 'ល', 'ែ', 'ង']","['ត', 'ំ', 'ប', 'ន', '់']","[1, 2, 3, 2, 4, 5, 2, 6, 2, 4, 5, 2, 7, 2, 4, ...","[1, 2, 14, 2, 4, 5, 2, 15, 2, 4, 5, 2, 16, 2, ..."
2,កន្លែង,ស្ថាន,1,"['ក', 'ន', '្', 'ល', 'ែ', 'ង']","['ស', '្', 'ថ', 'ា', 'ន']","[1, 2, 3, 2, 4, 5, 2, 6, 2, 4, 5, 2, 7, 2, 4, ...","[1, 2, 21, 2, 4, 5, 2, 7, 2, 4, 5, 2, 59, 2, 4..."
3,ទី,តំបន់,1,"['ទ', 'ី']","['ត', 'ំ', 'ប', 'ន', '់']","[1, 2, 12, 2, 4, 5, 2, 13, 2, 11]","[1, 2, 14, 2, 4, 5, 2, 15, 2, 4, 5, 2, 16, 2, ..."
4,ទី,ស្ថាន,1,"['ទ', 'ី']","['ស', '្', 'ថ', 'ា', 'ន']","[1, 2, 12, 2, 4, 5, 2, 13, 2, 11]","[1, 2, 21, 2, 4, 5, 2, 7, 2, 4, 5, 2, 59, 2, 4..."


In [9]:
df = df.sample(frac=1, random_state=42).reset_index(drop=True)


In [10]:
df.head()

,word1,word2,similarity,chars1,chars2,seq1,seq2
0,អរិន្ទ,គូវិវាទ,1,"['អ', 'រ', 'ិ', 'ន', '្', 'ទ']","['គ', 'ូ', 'វ', 'ិ', 'វ', 'ា', 'ទ']","[1, 2, 23, 2, 4, 5, 2, 22, 2, 4, 5, 2, 26, 2, ...","[1, 2, 37, 2, 4, 5, 2, 38, 2, 4, 5, 2, 44, 2, ..."
1,ព្រឹក្សា,អឍ្ឍរាត្រី,0,"['ព', '្', 'រ', 'ឹ', 'ក', '្', 'ស', 'ា']","['អ', 'ឍ', '្', 'ឍ', 'រ', 'ា', 'ត', '្', 'រ', ...","[1, 2, 29, 2, 4, 5, 2, 7, 2, 4, 5, 2, 22, 2, 4...","[1, 2, 23, 2, 4, 5, 2, 76, 2, 4, 5, 2, 7, 2, 4..."
2,ផ្ទំ,ព្រះកេសា,0,"['ផ', '្', 'ទ', 'ំ']","['ព', '្', 'រ', 'ះ', 'ក', 'េ', 'ស', 'ា']","[1, 2, 65, 2, 4, 5, 2, 7, 2, 4, 5, 2, 12, 2, 4...","[1, 2, 29, 2, 4, 5, 2, 7, 2, 4, 5, 2, 22, 2, 4..."
3,កុលបុត្រ,មិត្ត,0,"['ក', 'ុ', 'ល', 'ប', 'ុ', 'ត', '្', 'រ']","['ម', 'ិ', 'ត', '្', 'ត']","[1, 2, 3, 2, 4, 5, 2, 42, 2, 4, 5, 2, 8, 2, 4,...","[1, 2, 39, 2, 4, 5, 2, 26, 2, 4, 5, 2, 14, 2, ..."
4,គណបក្ខ,គណបក្ស,1,"['គ', 'ណ', 'ប', 'ក', '្', 'ខ']","['គ', 'ណ', 'ប', 'ក', '្', 'ស']","[1, 2, 37, 2, 4, 5, 2, 20, 2, 4, 5, 2, 16, 2, ...","[1, 2, 37, 2, 4, 5, 2, 20, 2, 4, 5, 2, 16, 2, ..."


In [11]:
# ----------------------------
# Step 4: Pad sequences
# ----------------------------
max_len = max(df['seq1'].map(len).max(), df['seq2'].map(len).max())

X1 = pad_sequences(
    df['seq1'],
    maxlen=max_len,
    padding="post",   # ✅ RIGHT padding
    truncating="post"
)

X2 = pad_sequences(
    df['seq2'],
    maxlen=max_len,
    padding="post",
    truncating="post"
)
y = df['similarity'].values


In [12]:
max_len

100

In [13]:
from sklearn.model_selection import train_test_split

X1_train, X1_test, X2_train, X2_test, y_train, y_test = train_test_split(
    X1, X2, y,
    test_size=0.2,
    random_state=42,
)


In [14]:
embed_dim = 64

inp1 = Input(shape=(max_len,))
inp2 = Input(shape=(max_len,))

embedding = Embedding(
    input_dim=vocab_size,
    output_dim=embed_dim,
    mask_zero=True
)

rnn = LSTM(64)

v1 = rnn(embedding(inp1))
v2 = rnn(embedding(inp2))

sim = Dot(axes=1, normalize=True)([v1, v2])

out = Dense(1, activation="sigmoid")(sim)

model = Model([inp1, inp2], out)
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


I0000 00:00:1766321243.382702      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 100)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 100)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 100, 64)   │      4,992 │ input_layer[0][0… │
│ (Embedding)         │                   │            │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 100)       │          0 │ input_layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, 100)       │          0 │ input_layer_1[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 64)        │     33,024 │ embedding[0][0],  │
│                     │                   │            │ not_equal[0][0],  │
│                     │                   │            │ embedding[1][0],  │
│                     │                   │            │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dot (Dot)           │ (None, 1)         │          0 │ lstm[0][0],       │
│                     │                   │            │ lstm[1][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 1)         │          2 │ dot[0][0]         │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 38,018 (148.51 KB)

 Trainable params: 38,018 (148.51 KB)

 Non-trainable params: 0 (0.00 B)

In [15]:
model.fit(
    [X1_train, X2_train],
    y_train,
    epochs=20,
    batch_size=8,
    validation_split=0.1
)


Epoch 1/20


I0000 00:00:1766321248.121297      66 cuda_dnn.cc:529] Loaded cuDNN version 91002


1080/1080 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - accuracy: 0.7455 - loss: 0.5683 - val_accuracy: 0.7740 - val_loss: 0.5368
Epoch 2/20
1080/1080 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.7544 - loss: 0.5577 - val_accuracy: 0.7740 - val_loss: 0.5368
Epoch 3/20
1080/1080 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.7425 - loss: 0.5696 - val_accuracy: 0.7719 - val_loss: 0.5343
Epoch 4/20
1080/1080 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.7446 - loss: 0.5666 - val_accuracy: 0.7740 - val_loss: 0.5290
Epoch 5/20
1080/1080 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.7412 - loss: 0.5661 - val_accuracy: 0.7760 - val_loss: 0.5277
Epoch 6/20
1080/1080 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.7456 - loss: 0.5501 - val_accuracy: 0.7792 - val_loss: 0.5228
Epoch 7/20
1080/1080 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.7692 - loss: 0.5234 - val_accuracy: 0.7667 - val_loss: 0.5239
Epoch 8/20
1080/1080 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.7604 - loss: 0.5283 - val_accura

In [16]:
model.save("/kaggle/working/khmer_synonym_rnn_model.h5")


In [17]:
test_loss, test_acc = model.evaluate([X1_test, X2_test], y_test)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_acc)


75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7626 - loss: 0.5207
Test Loss: 0.533618688583374
Test Accuracy: 0.7516666650772095


In [18]:
from sklearn.metrics import classification_report


y_pred = (model.predict([X1_test, X2_test]) > 0.5).astype(int)

print(classification_report(y_test, y_pred))


75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
              precision    recall  f1-score   support

           0       0.78      0.91      0.84      1737
           1       0.59      0.33      0.42       663

    accuracy                           0.75      2400
   macro avg       0.69      0.62      0.63      2400
weighted avg       0.73      0.75      0.73      2400



In [19]:
def predict_similarity(word1, word2):
    """Predict similarity between two Khmer words"""
    # Use EXACT SAME padding as training
    seq1 = pad_sequences(
        [encode(list(word1))],
        maxlen=max_len,
        padding="post",
        truncating="post"
    )
    seq2 = pad_sequences(
        [encode(list(word2))],
        maxlen=max_len,
        padding="post",
        truncating="post"
    )
    
    score = model.predict([seq1, seq2], verbose=0)[0][0]
    return float(score)

def is_similar(word1, word2, threshold=0.9):
    """Check if words are similar (teacher's threshold: >0.9)"""
    score = predict_similarity(word1, word2)
    return score, score >= threshold

In [20]:
# Test cases
test_pairs = [
    ("អាយុ", "ជន្ម"),      # Should be similar
    ("ទី", "ស្រីកល្យាណី"),  # Should NOT be similar
    ("ច័ន្ទ", "ខែ"),        # Should be similar (moon)
    ("ទឹក", "អាង")         # Should NOT be similar (water vs pond)
]

for w1, w2 in test_pairs:
    score, decision = is_similar(w1, w2)
    print(f"{w1} vs {w2}: {score:.4f} → {'Similar' if decision else 'Not similar'}")

អាយុ vs ជន្ម: 0.6582 → Not similar
ទី vs ស្រីកល្យាណី: 0.6928 → Not similar
ច័ន្ទ vs ខែ: 0.4205 → Not similar
ទឹក vs អាង: 0.4090 → Not similar


# **Create experiment tracking table**

In [21]:
# Before tracking experiments, make sure you trained and saved history
print("Checking training history...")

# If you already trained but didn't save history, check what you have
if 'history' not in locals():
    print("'history' variable not found!")
    print("\nAvailable variables starting with 'h':")
    for var_name in dir():
        if var_name.startswith('h'):
            print(f"  - {var_name}")
    
    # Check if you have any training history saved
    if 'hist' in locals():
        history = hist
        print(f"\nFound 'hist', using it as 'history'")
    elif 'model_history' in locals():
        history = model_history
        print(f"\nFound 'model_history', using it as 'history'")
    else:
        print("\nNo history found! You need to train the model first.")

Checking training history...
'history' variable not found!

Available variables starting with 'h':

No history found! You need to train the model first.


# **Full model architecture code**

In [22]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, Lambda, Dense,
    SimpleRNN, LSTM, GRU, Dropout, BatchNormalization,
    Bidirectional, GlobalMaxPooling1D, concatenate
)
from tensorflow.keras import backend as K
import numpy as np

class SiameseKhmerSimilarity:
    """
    Complete Siamese Network for Khmer Word Similarity
    Supports: SimpleRNN, LSTM, GRU variants – now with fixed advanced model
    """
    
    def __init__(self, vocab_size, max_len, embed_dim=64, rnn_units=64):
        """
        Initialize the Siamese model
        
        Args:
            vocab_size: Size of character vocabulary
            max_len: Maximum sequence length
            embed_dim: Embedding dimension
            rnn_units: Number of units in RNN layer
        """
        self.vocab_size = vocab_size
        self.max_len = max_len
        self.embed_dim = embed_dim
        self.rnn_units = rnn_units
        
        # Character mapping (should be loaded from pickle)
        self.char2idx = None
        self.idx2char = None
        
    def create_embedding_layer(self):
        """Create shared embedding layer for both inputs"""
        return Embedding(
            input_dim=self.vocab_size + 1,  # +1 for padding token
            output_dim=self.embed_dim,
            input_length=self.max_len,
            mask_zero=True,
            name="shared_embedding"
        )
    
    def create_rnn_layer(self, rnn_type='lstm', return_sequences=False):
        """
        Create RNN layer based on specified type
        """
        if rnn_type.lower() == 'lstm':
            return LSTM(
                self.rnn_units,
                return_sequences=return_sequences,
                dropout=0.2,
                recurrent_dropout=0.2,
                name="shared_lstm"
            )
        elif rnn_type.lower() == 'gru':
            return GRU(
                self.rnn_units,
                return_sequences=return_sequences,
                dropout=0.2,
                recurrent_dropout=0.2,
                name="shared_gru"
            )
        elif rnn_type.lower() == 'rnn':
            return SimpleRNN(
                self.rnn_units,
                return_sequences=return_sequences,
                dropout=0.2,
                recurrent_dropout=0.2,
                name="shared_rnn"
            )
        else:
            raise ValueError(f"Unknown RNN type: {rnn_type}")
    
    def create_siamese_model(self, rnn_type='lstm', use_attention=False):
        """
        Create complete Siamese model (your original simple version)
        """
        input_a = Input(shape=(self.max_len,), name="word_input_a")
        input_b = Input(shape=(self.max_len,), name="word_input_b")
        
        embedding_layer = self.create_embedding_layer()
        embedded_a = embedding_layer(input_a)
        embedded_b = embedding_layer(input_b)
        
        rnn_layer = self.create_rnn_layer(rnn_type=rnn_type, return_sequences=False)
        
        encoded_a = rnn_layer(embedded_a)
        encoded_b = rnn_layer(embedded_b)
        
        encoded_a = BatchNormalization(name="bn_a")(encoded_a)
        encoded_b = BatchNormalization(name="bn_b")(encoded_b)
        
        similarity = Lambda(
            lambda x: K.abs(x[0] - x[1]),
            name="manhattan_distance"
        )([encoded_a, encoded_b])
        
        if use_attention:
            attention = Dense(self.rnn_units, activation='tanh', name="attention")(similarity)
            attention = tf.keras.layers.Attention()([attention, attention])
            similarity = tf.keras.layers.Concatenate()([similarity, attention])
        
        dense1 = Dense(64, activation='relu', name="dense_1")(similarity)
        dense1 = Dropout(0.3, name="dropout_1")(dense1)
        
        dense2 = Dense(32, activation='relu', name="dense_2")(dense1)
        dense2 = Dropout(0.3, name="dropout_2")(dense2)
        
        output = Dense(1, activation='sigmoid', name="similarity_output")(dense2)
        
        model = Model(
            inputs=[input_a, input_b],
            outputs=output,
            name=f"Siamese_{rnn_type.upper()}_Khmer_Similarity"
        )
        
        return model
    
    def create_advanced_siamese_model(self, rnn_type='lstm'):
        """
        Advanced version with bidirectional RNN + multiple similarity measures
        Shape mismatch is completely fixed – ready to train!
        """
        # Input layers
        input_a = Input(shape=(self.max_len,), name="word_input_a")
        input_b = Input(shape=(self.max_len,), name="word_input_b")
        
        # Shared embedding
        embedding_layer = self.create_embedding_layer()
        embedded_a = embedding_layer(input_a)
        embedded_b = embedding_layer(input_b)
        
        # Choose RNN cell
        if rnn_type.lower() == 'lstm':
            rnn_cell = LSTM
        elif rnn_type.lower() == 'gru':
            rnn_cell = GRU
        else:
            rnn_cell = SimpleRNN
        
        # Bidirectional layer
        bidirectional_rnn = Bidirectional(
            rnn_cell(self.rnn_units, return_sequences=True),
            name="bidirectional_rnn"
        )
        
        # Encode sequences
        encoded_a = bidirectional_rnn(embedded_a)
        encoded_b = bidirectional_rnn(embedded_b)
        
        # Global max pooling
        pooled_a = GlobalMaxPooling1D(name="pool_a")(encoded_a)
        pooled_b = GlobalMaxPooling1D(name="pool_b")(encoded_b)
        
        # Manhattan distance
        manhattan = Lambda(
            lambda x: K.abs(x[0] - x[1]),
            name="manhattan"
        )([pooled_a, pooled_b])
        
        # Euclidean distance
        euclidean = Lambda(
            lambda x: K.sqrt(K.sum(K.square(x[0] - x[1]), axis=-1, keepdims=True) + K.epsilon()),
            name="euclidean"
        )([pooled_a, pooled_b])
        
        # Cosine similarity – fixed to always return shape (batch, 1)
        def cosine_fixed(x):
            a, b = x
            a_norm = K.l2_normalize(a, axis=-1)
            b_norm = K.l2_normalize(b, axis=-1)
            return K.sum(a_norm * b_norm, axis=-1, keepdims=True)
        
        cosine = Lambda(cosine_fixed, name="cosine_similarity")([pooled_a, pooled_b])
        
        # Combine all (now all have shape (batch, features))
        combined = concatenate([manhattan, euclidean, cosine], name="combined_similarities")
        
        # Dense layers
        dense1 = Dense(128, activation='relu', name="dense1")(combined)
        dense1 = Dropout(0.4)(dense1)
        dense1 = BatchNormalization()(dense1)
        
        dense2 = Dense(64, activation='relu', name="dense2")(dense1)
        dense2 = Dropout(0.3)(dense2)
        
        dense3 = Dense(32, activation='relu', name="dense3")(dense2)
        dense3 = Dropout(0.2)(dense3)
        
        # Output
        output = Dense(1, activation='sigmoid', name="output")(dense3)
        
        model = Model(
            inputs=[input_a, input_b],
            outputs=output,
            name=f"Advanced_Siamese_{rnn_type.upper()}"
        )
        
        return model
    
    def compile_model(self, model, learning_rate=0.001):
        """Compile model with appropriate loss and metrics"""
        optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
        
        model.compile(
            optimizer=optimizer,
            loss='binary_crossentropy',
            metrics=[
                'accuracy',
                tf.keras.metrics.Precision(name='precision'),
                tf.keras.metrics.Recall(name='recall'),
                tf.keras.metrics.AUC(name='auc')
            ]
        )
        
        return model
    
    def model_summary(self, model):
        """Print enhanced model summary"""
        print("=" * 80)
        print(f"Model: {model.name}")
        print("=" * 80)
        model.summary()
        
        trainable_params = np.sum([K.count_params(w) for w in model.trainable_weights])
        non_trainable_params = np.sum([K.count_params(w) for w in model.non_trainable_weights])
        
        print(f"\nTotal parameters: {trainable_params + non_trainable_params:,}")
        print(f"Trainable parameters: {trainable_params:,}")
        print(f"Non-trainable parameters: {non_trainable_params:,}")
        print("=" * 80)

In [23]:
# Initialize parameters
VOCAB_SIZE = len(char2idx)  # Your actual vocabulary size
MAX_LEN = max_len  # Your actual max length

# Create model manager
model_manager = SiameseKhmerSimilarity(
    vocab_size=VOCAB_SIZE,
    max_len=MAX_LEN,
    embed_dim=128,  # Experiment with 32, 64, 128
    rnn_units=64    # Experiment with 32, 64, 128
)

# Create different model variants
print("Creating Simple RNN model...")
simple_rnn_model = model_manager.create_siamese_model(rnn_type='rnn')
simple_rnn_model = model_manager.compile_model(simple_rnn_model)
model_manager.model_summary(simple_rnn_model)

print("\n" + "="*80 + "\n")

print("Creating LSTM model...")
lstm_model = model_manager.create_siamese_model(rnn_type='lstm')
lstm_model = model_manager.compile_model(lstm_model)
model_manager.model_summary(lstm_model)

print("\n" + "="*80 + "\n")

print("Creating GRU model...")
gru_model = model_manager.create_siamese_model(rnn_type='gru')
gru_model = model_manager.compile_model(gru_model)
model_manager.model_summary(gru_model)

print("\n" + "="*80 + "\n")

print("Creating Advanced LSTM model...")
advanced_lstm_model = model_manager.create_advanced_siamese_model(rnn_type='lstm')
advanced_lstm_model = model_manager.compile_model(advanced_lstm_model)
model_manager.model_summary(advanced_lstm_model)

Creating Simple RNN model...
Model: Siamese_RNN_Khmer_Similarity


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "Siamese_RNN_Khmer_Similarity"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ word_input_a        │ (None, 100)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ word_input_b        │ (None, 100)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_embedding    │ (None, 100, 128)  │      9,984 │ word_input_a[0][… │
│ (Embedding)         │                   │            │ word_input_b[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_2         │ (None, 100)       │          0 │ word_input_a[0][… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_3         │ (None, 100)       │          0 │ word_input_b[0][… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_rnn          │ (None, 64)        │     12,352 │ shared_embedding… │
│ (SimpleRNN)         │                   │            │ not_equal_2[0][0… │
│                     │                   │            │ shared_embedding… │
│                     │                   │            │ not_equal_3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_a                │ (None, 64)        │        256 │ shared_rnn[0][0]  │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_b                │ (None, 64)        │        256 │ shared_rnn[1][0]  │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ manhattan_distance  │ (None, 64)        │          0 │ bn_a[0][0],       │
│ (Lambda)            │                   │            │ bn_b[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      4,160 │ manhattan_distan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32)        │      2,080 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 32)        │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ similarity_output   │ (None, 1)         │         33 │ dropout_2[0][0]   │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 29,121 (113.75 KB)

 Trainable params: 28,865 (112.75 KB)

... (17 outputs left out)

 Total params: 136,449 (533.00 KB)

 Trainable params: 136,193 (532.00 KB)

 Non-trainable params: 256 (1.00 KB)


Total parameters: 136,449
Trainable parameters: 136,193
Non-trainable parameters: 256


In [24]:
import tensorflow as tf
from sklearn.model_selection import train_test_split
import os
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ==================== DATA SPLIT ====================
print("Splitting data into train/test sets...")
X1_train, X1_test, X2_train, X2_test, y_train, y_test = train_test_split(
    X1, X2, y,
    test_size=0.2,
    random_state=42,
)

print(f"Training samples: {len(y_train)}")
print(f"Test samples: {len(y_test)}")
print(f"Similar pair ratio (train): {y_train.mean():.4f}")
print(f"Similar pair ratio (test): {y_test.mean():.4f}")

# ==================== RE-USABLE TRAINING FUNCTION ====================
def train_model_variant(model, model_name, X1_train, X2_train, y_train, X1_test, X2_test, y_test,
                       epochs=50, batch_size=32):
    """
    Train a single model variant with best practices
    """
    print(f"\n{'='*80}")
    print(f"TRAINING: {model_name.upper()}")
    print(f"{'='*80}")
    
    # Create directory for saving if not exists
    os.makedirs("saved_models", exist_ok=True)
    
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_auc',
            mode='max',
            patience=10,
            restore_best_weights=True,
            verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_auc',
            mode='max',
            factor=0.5,
            patience=5,
            min_lr=1e-7,
            verbose=1
        ),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=f"saved_models/best_{model_name}.h5",
            monitor='val_auc',
            mode='max',
            save_best_only=True,
            verbose=1
        )
    ]
    
    history = model.fit(
        [X1_train, X2_train], y_train,
        validation_data=([X1_test, X2_test], y_test),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=1
    )
    
    # Final evaluation
    print(f"\n{'='*80}")
    print(f"FINAL EVALUATION - {model_name.upper()}")
    print(f"{'='*80}")
    eval_results = model.evaluate([X1_test, X2_test], y_test, verbose=0)
    val_loss, val_acc, val_prec, val_rec, val_auc = eval_results
    val_f1 = 2 * (val_prec * val_rec) / (val_prec + val_rec + 1e-8)
    best_val_auc = max(history.history['val_auc'])
    
    print(f"Test Loss:      {val_loss:.4f}")
    print(f"Test Accuracy:  {val_acc:.4f}")
    print(f"Test Precision: {val_prec:.4f}")
    print(f"Test Recall:    {val_rec:.4f}")
    print(f"Test AUC:       {val_auc:.4f}")
    print(f"Test F1:        {val_f1:.4f}")
    print(f"Best Val AUC:   {best_val_auc:.4f}")
    
    # Save final model as .h5
    model.save(f"saved_models/final_{model_name}.h5")
    print(f"\nModel saved: saved_models/best_{model_name}.h5 & final_{model_name}.h5")
    
    return history, model

# ==================== TRAIN ALL MODEL VARIANTS ====================
print("\n" + "="*100)
print("STARTING TRAINING FOR ALL MODEL VARIANTS")
print("="*100)

# 1. Simple RNN
history_rnn, trained_rnn = train_model_variant(
    model=simple_rnn_model,
    model_name="simple_rnn",
    X1_train=X1_train, X2_train=X2_train, y_train=y_train,
    X1_test=X1_test, X2_test=X2_test, y_test=y_test,
    epochs=50,
    batch_size=32
)

# 2. Basic LSTM
history_lstm, trained_lstm = train_model_variant(
    model=lstm_model,
    model_name="basic_lstm",
    X1_train=X1_train, X2_train=X2_train, y_train=y_train,
    X1_test=X1_test, X2_test=X2_test, y_test=y_test,
    epochs=50,
    batch_size=32
)

# 3. Basic GRU
history_gru, trained_gru = train_model_variant(
    model=gru_model,
    model_name="basic_gru",
    X1_train=X1_train, X2_train=X2_train, y_train=y_train,
    X1_test=X1_test, X2_test=X2_test, y_test=y_test,
    epochs=50,
    batch_size=32
)

# 4. Advanced LSTM (this should be the strongest!)
history_adv, trained_advanced = train_model_variant(
    model=advanced_lstm_model,
    model_name="advanced_lstm",
    X1_train=X1_train, X2_train=X2_train, y_train=y_train,
    X1_test=X1_test, X2_test=X2_test, y_test=y_test,
    epochs=50,
    batch_size=32
)

# ==================== TESTING UTILITIES ====================
def predict_similarity(model, word1, word2, char2idx, max_len):
    """Predict similarity between two Khmer words"""
    def encode(chars):
        return [char2idx.get(c, 0) for c in chars]
    
    # Use EXACT SAME padding as training
    seq1 = pad_sequences(
        [encode(list(word1))],
        maxlen=max_len,
        padding="post",
        truncating="post"
    )
    seq2 = pad_sequences(
        [encode(list(word2))],
        maxlen=max_len,
        padding="post",
        truncating="post"
    )
    
    score = model.predict([seq1, seq2], verbose=0)[0][0]
    return float(score)

def is_similar(model, word1, word2, char2idx, max_len, threshold=0.9):
    """Check if words are similar (teacher's threshold: >0.9)"""
    score = predict_similarity(model, word1, word2, char2idx, max_len)
    return score, score >= threshold

def test_model_on_examples(model, model_name, test_pairs, char2idx, max_len):
    """Test a model on example word pairs"""
    print(f"\n{'='*80}")
    print(f"TESTING MODEL: {model_name.upper()}")
    print(f"{'='*80}")
    
    for w1, w2 in test_pairs:
        score, decision = is_similar(model, w1, w2, char2idx, max_len)
        print(f"{w1:15s} vs {w2:15s}: {score:.4f} → {'✓ Similar' if decision else '✗ Not similar'}")
    print()

# ==================== TEST ALL MODELS ====================
print("\n" + "="*100)
print("TESTING ALL MODELS ON EXAMPLE PAIRS")
print("="*100)

# Test cases
test_pairs = [
    ("អាយុ", "ជន្ម"),      # Should be similar (age/birth)
    ("ទី", "ស្រីកល្យាណី"),  # Should NOT be similar
    ("ច័ន្ទ", "ខែ"),        # Should be similar (moon/month)
    ("ទឹក", "អាង"),         # Should NOT be similar (water vs pond)
]

# Test all trained models
test_model_on_examples(trained_rnn, "Simple RNN", test_pairs, char2idx, max_len)
test_model_on_examples(trained_lstm, "Basic LSTM", test_pairs, char2idx, max_len)
test_model_on_examples(trained_gru, "Basic GRU", test_pairs, char2idx, max_len)
test_model_on_examples(trained_advanced, "Advanced LSTM", test_pairs, char2idx, max_len)

# ==================== SUMMARY ====================
print("\n" + "="*100)
print("TRAINING AND TESTING COMPLETED!")
print("="*100)
print("\nSaved Models:")
print("  • saved_models/best_simple_rnn.h5")
print("  • saved_models/final_simple_rnn.h5")
print("  • saved_models/best_basic_lstm.h5")
print("  • saved_models/final_basic_lstm.h5")
print("  • saved_models/best_basic_gru.h5")
print("  • saved_models/final_basic_gru.h5")
print("  • saved_models/best_advanced_lstm.h5")
print("  • saved_models/final_advanced_lstm.h5")
print("\nRecommendation: Use 'best_advanced_lstm.h5' for production!")
print("="*100)

# ==================== HOW TO LOAD AND USE LATER ====================
print("\n" + "="*100)
print("HOW TO LOAD AND USE SAVED MODELS")
print("="*100)
print("""
# Load a saved model:
from tensorflow.keras.models import load_model

model = load_model('saved_models/best_advanced_lstm.h5')

# Make predictions:
score = predict_similarity(model, "អាយុ", "ជន្ម", char2idx, max_len)
print(f"Similarity score: {score:.4f}")

# Check if similar:
score, is_sim = is_similar(model, "អាយុ", "ជន្ម", char2idx, max_len, threshold=0.9)
print(f"Similar: {is_sim}")
""")

Splitting data into train/test sets...
Training samples: 9598
Test samples: 2400
Similar pair ratio (train): 0.2526
Similar pair ratio (test): 0.2762

STARTING TRAINING FOR ALL MODEL VARIANTS

TRAINING: SIMPLE_RNN
Epoch 1/50


I0000 00:00:1766321447.618358      66 service.cc:152] XLA service 0x7dd2c8146530 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1766321447.618424      66 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0


  7/300 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - accuracy: 0.5176 - auc: 0.5266 - loss: 0.8294 - precision: 0.2359 - recall: 0.4695

I0000 00:00:1766321452.663588      66 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


300/300 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.6938 - auc: 0.4983 - loss: 0.6486 - precision: 0.2568 - recall: 0.1020
Epoch 1: val_auc improved from -inf to 0.50081, saving model to saved_models/best_simple_rnn.h5


300/300 ━━━━━━━━━━━━━━━━━━━━ 21s 41ms/step - accuracy: 0.6939 - auc: 0.4983 - loss: 0.6484 - precision: 0.2569 - recall: 0.1019 - val_accuracy: 0.7237 - val_auc: 0.5008 - val_loss: 0.6009 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 0.0010
Epoch 2/50
298/300 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.7378 - auc: 0.4981 - loss: 0.5933 - precision: 0.2776 - recall: 0.0015
Epoch 2: val_auc improved from 0.50081 to 0.51323, saving model to saved_models/best_simple_rnn.h5


... (91 outputs left out)

300/300 ━━━━━━━━━━━━━━━━━━━━ 6s 21ms/step - accuracy: 0.9179 - auc: 0.9520 - loss: 0.2242 - precision: 0.8920 - recall: 0.7726 - val_accuracy: 0.8525 - val_auc: 0.8865 - val_loss: 0.3721 - val_precision: 0.7456 - val_recall: 0.7074 - learning_rate: 0.0010
Epoch 17/50
298/300 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9246 - auc: 0.9569 - loss: 0.2091 - precision: 0.8840 - recall: 0.8091
Epoch 17: val_auc did not improve from 0.88651
300/300 ━━━━━━━━━━━━━━━━━━━━ 6s 21ms/step - accuracy: 0.9246 - auc: 0.9568 - loss: 0.2091 - precision: 0.8839 - recall: 0.8090 - val_accuracy: 0.8542 - val_auc: 0.8806 - val_loss: 0.4023 - val_precision: 0.7882 - val_recall: 0.6456 - learning_rate: 0.0010
Epoch 18/50
298/300 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9258 - auc: 0.9613 - loss: 0.1985 - precision: 0.9024 - recall: 0.7957
Epoch 18: val_auc did not improve from 0.88651
300/300 ━━━━━━━━━━━━━━━━━━━━ 6s 21ms/step - accuracy: 0.9257 - auc: 0.9612 - loss: 0.1986 - precision: 0.9023 - reca

300/300 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - accuracy: 0.9662 - auc: 0.9903 - loss: 0.0955 - precision: 0.9440 - recall: 0.9179 - val_accuracy: 0.8479 - val_auc: 0.8884 - val_loss: 0.5000 - val_precision: 0.7307 - val_recall: 0.7119 - learning_rate: 5.0000e-04
Epoch 27/50
298/300 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9707 - auc: 0.9910 - loss: 0.0898 - precision: 0.9549 - recall: 0.9283
Epoch 27: val_auc improved from 0.88835 to 0.89296, saving model to saved_models/best_advanced_lstm.h5


300/300 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - accuracy: 0.9707 - auc: 0.9910 - loss: 0.0899 - precision: 0.9548 - recall: 0.9282 - val_accuracy: 0.8525 - val_auc: 0.8930 - val_loss: 0.5113 - val_precision: 0.7262 - val_recall: 0.7481 - learning_rate: 5.0000e-04
Epoch 28/50
300/300 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9698 - auc: 0.9932 - loss: 0.0845 - precision: 0.9612 - recall: 0.9205
Epoch 28: val_auc did not improve from 0.89296
300/300 ━━━━━━━━━━━━━━━━━━━━ 6s 21ms/step - accuracy: 0.9698 - auc: 0.9932 - loss: 0.0845 - precision: 0.9612 - recall: 0.9205 - val_accuracy: 0.8413 - val_auc: 0.8929 - val_loss: 0.5668 - val_precision: 0.6841 - val_recall: 0.7903 - learning_rate: 5.0000e-04
Epoch 29/50
298/300 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9699 - auc: 0.9933 - loss: 0.0840 - precision: 0.9523 - recall: 0.9277
Epoch 29: val_auc did not improve from 0.89296
300/300 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - accuracy: 0.9698 - auc: 0.9933 - loss: 0.0841 - precision: 0.952

Test Loss:      0.5113
Test Accuracy:  0.8525
Test Precision: 0.7262
Test Recall:    0.7481
Test AUC:       0.8930
Test F1:        0.7370
Best Val AUC:   0.8930

Model saved: saved_models/best_advanced_lstm.h5 & final_advanced_lstm.h5

TESTING ALL MODELS ON EXAMPLE PAIRS

TESTING MODEL: SIMPLE RNN
អាយុ            vs ជន្ម           : 0.3738 → ✗ Not similar
ទី              vs ស្រីកល្យាណី    : 0.3051 → ✗ Not similar
ច័ន្ទ           vs ខែ             : 0.3514 → ✗ Not similar
ទឹក             vs អាង            : 0.3768 → ✗ Not similar


TESTING MODEL: BASIC LSTM
អាយុ            vs ជន្ម           : 0.2789 → ✗ Not similar
ទី              vs ស្រីកល្យាណី    : 0.9951 → ✓ Similar
ច័ន្ទ           vs ខែ             : 0.9037 → ✓ Similar
ទឹក             vs អាង            : 0.9794 → ✓ Similar


TESTING MODEL: BASIC GRU
អាយុ            vs ជន្ម           : 0.0378 → ✗ Not similar
ទី              vs ស្រីកល្យាណី    : 0.7246 → ✗ Not similar
ច័ន្ទ           vs ខែ             : 0.1082 → ✗ Not similar
ទឹក      

**Constrasive Learning**

In [25]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, Lambda, Dense,
    SimpleRNN, LSTM, GRU, Dropout, BatchNormalization,
    Bidirectional, GlobalMaxPooling1D, GlobalAveragePooling1D,
    concatenate, Add, Multiply
)
from tensorflow.keras import backend as K
import numpy as np

class ImprovedSiameseKhmerSimilarity:
    """
    Improved Siamese Network with Contrastive Learning
    Uses proper distance-based loss for better similarity learning
    """
    
    def __init__(self, vocab_size, max_len, embed_dim=128, rnn_units=128):
        """
        Initialize the Siamese model
        
        Args:
            vocab_size: Size of character vocabulary
            max_len: Maximum sequence length
            embed_dim: Embedding dimension (increased to 128)
            rnn_units: Number of units in RNN layer (increased to 128)
        """
        self.vocab_size = vocab_size
        self.max_len = max_len
        self.embed_dim = embed_dim
        self.rnn_units = rnn_units
        self.margin = 1.0  # Margin for contrastive loss
        
    # ==================== CONTRASTIVE LOSS ====================
    def contrastive_loss(self, y_true, y_pred, margin=1.0):
        """
        Contrastive loss function
        
        For similar pairs (y=1): minimize distance
        For dissimilar pairs (y=0): maximize distance up to margin
        
        Loss = y * distance^2 + (1-y) * max(margin - distance, 0)^2
        """
        square_pred = K.square(y_pred)
        margin_square = K.square(K.maximum(margin - y_pred, 0))
        return K.mean(y_true * square_pred + (1 - y_true) * margin_square)
    
    # ==================== TRIPLET LOSS (Alternative) ====================
    def triplet_loss(self, y_true, y_pred, margin=0.2):
        """
        Triplet loss: anchor should be closer to positive than negative
        """
        return K.maximum(y_pred + margin, 0.0)
    
    # ==================== EMBEDDING LAYERS ====================
    def create_embedding_layer(self):
        """Create shared embedding layer"""
        return Embedding(
            input_dim=self.vocab_size + 1,
            output_dim=self.embed_dim,
            input_length=self.max_len,
            mask_zero=True,
            embeddings_initializer='glorot_uniform',
            name="shared_embedding"
        )
    
    # ==================== ENCODER NETWORK ====================
    def create_encoder(self, rnn_type='lstm'):
        """
        Create the shared encoder network
        This extracts features from input sequences
        """
        if rnn_type.lower() == 'lstm':
            rnn_cell = LSTM
        elif rnn_type.lower() == 'gru':
            rnn_cell = GRU
        else:
            rnn_cell = SimpleRNN
        
        def encoder(input_seq):
            # Embedding
            embedding_layer = self.create_embedding_layer()
            embedded = embedding_layer(input_seq)
            
            # Bidirectional RNN
            bi_rnn = Bidirectional(
                rnn_cell(self.rnn_units, return_sequences=True, dropout=0.3)
            )(embedded)
            
            # Both max and average pooling
            max_pool = GlobalMaxPooling1D()(bi_rnn)
            avg_pool = GlobalAveragePooling1D()(bi_rnn)
            
            # Concatenate pooling outputs
            combined = concatenate([max_pool, avg_pool])
            
            # Dense projection
            dense1 = Dense(256, activation='relu')(combined)
            dense1 = BatchNormalization()(dense1)
            dense1 = Dropout(0.4)(dense1)
            
            dense2 = Dense(128, activation='relu')(dense1)
            dense2 = BatchNormalization()(dense2)
            
            # L2 normalize the output (important for contrastive learning!)
            normalized = Lambda(lambda x: K.l2_normalize(x, axis=1))(dense2)
            
            return normalized
        
        return encoder
    
    # ==================== MODEL 1: CONTRASTIVE SIAMESE ====================
    def create_contrastive_siamese(self, rnn_type='lstm'):
        """
        Siamese network with contrastive loss
        Outputs: Euclidean distance between embeddings
        """
        input_a = Input(shape=(self.max_len,), name="input_a")
        input_b = Input(shape=(self.max_len,), name="input_b")
        
        # Shared embedding
        embedding_layer = Embedding(
            input_dim=self.vocab_size + 1,
            output_dim=self.embed_dim,
            input_length=self.max_len,
            mask_zero=True,
            embeddings_initializer='glorot_uniform',
            name="shared_embedding"
        )
        
        # RNN cell
        if rnn_type.lower() == 'lstm':
            rnn_cell = LSTM
        elif rnn_type.lower() == 'gru':
            rnn_cell = GRU
        else:
            rnn_cell = SimpleRNN
        
        # Bidirectional RNN
        bi_rnn = Bidirectional(
            rnn_cell(self.rnn_units, return_sequences=True, dropout=0.3),
            name="shared_bi_rnn"
        )
        
        # Process both inputs
        embedded_a = embedding_layer(input_a)
        embedded_b = embedding_layer(input_b)
        
        rnn_a = bi_rnn(embedded_a)
        rnn_b = bi_rnn(embedded_b)
        
        # Pooling
        max_pool_a = GlobalMaxPooling1D(name="max_pool_a")(rnn_a)
        avg_pool_a = GlobalAveragePooling1D(name="avg_pool_a")(rnn_a)
        
        max_pool_b = GlobalMaxPooling1D(name="max_pool_b")(rnn_b)
        avg_pool_b = GlobalAveragePooling1D(name="avg_pool_b")(rnn_b)
        
        combined_a = concatenate([max_pool_a, avg_pool_a])
        combined_b = concatenate([max_pool_b, avg_pool_b])
        
        # Dense layers
        dense1 = Dense(256, activation='relu', name="shared_dense1")
        bn1 = BatchNormalization(name="shared_bn1")
        dropout1 = Dropout(0.4, name="shared_dropout1")
        
        processed_a = dropout1(bn1(dense1(combined_a)))
        processed_b = dropout1(bn1(dense1(combined_b)))
        
        dense2 = Dense(128, activation='relu', name="shared_dense2")
        bn2 = BatchNormalization(name="shared_bn2")
        
        embedding_a = bn2(dense2(processed_a))
        embedding_b = bn2(dense2(processed_b))
        
        # L2 normalize
        normalized_a = Lambda(lambda x: K.l2_normalize(x, axis=1), name="norm_a")(embedding_a)
        normalized_b = Lambda(lambda x: K.l2_normalize(x, axis=1), name="norm_b")(embedding_b)
        
        # Calculate Euclidean distance
        distance = Lambda(
            lambda x: K.sqrt(K.sum(K.square(x[0] - x[1]), axis=1, keepdims=True) + K.epsilon()),
            name="euclidean_distance"
        )([normalized_a, normalized_b])
        
        model = Model(
            inputs=[input_a, input_b],
            outputs=distance,
            name=f"Contrastive_Siamese_{rnn_type.upper()}"
        )
        
        return model
    
    # ==================== MODEL 2: HYBRID SIAMESE ====================
    def create_hybrid_siamese(self, rnn_type='lstm'):
        """
        Hybrid model: Uses distance + additional features
        More robust than pure distance-based
        """
        input_a = Input(shape=(self.max_len,), name="input_a")
        input_b = Input(shape=(self.max_len,), name="input_b")
        
        # Create shared layers
        embedding_layer = Embedding(
            input_dim=self.vocab_size + 1,
            output_dim=self.embed_dim,
            input_length=self.max_len,
            mask_zero=True,
            embeddings_initializer='glorot_uniform',
            name="shared_embedding"
        )
        
        # RNN cell
        if rnn_type.lower() == 'lstm':
            rnn_cell = LSTM
        elif rnn_type.lower() == 'gru':
            rnn_cell = GRU
        else:
            rnn_cell = SimpleRNN
        
        # Bidirectional RNN
        bi_rnn = Bidirectional(
            rnn_cell(self.rnn_units, return_sequences=True, dropout=0.3),
            name="shared_bi_rnn"
        )
        
        # Process both inputs through shared layers
        embedded_a = embedding_layer(input_a)
        embedded_b = embedding_layer(input_b)
        
        rnn_a = bi_rnn(embedded_a)
        rnn_b = bi_rnn(embedded_b)
        
        # Pooling
        max_pool_a = GlobalMaxPooling1D(name="max_pool_a")(rnn_a)
        avg_pool_a = GlobalAveragePooling1D(name="avg_pool_a")(rnn_a)
        
        max_pool_b = GlobalMaxPooling1D(name="max_pool_b")(rnn_b)
        avg_pool_b = GlobalAveragePooling1D(name="avg_pool_b")(rnn_b)
        
        # Combine pooling
        combined_a = concatenate([max_pool_a, avg_pool_a], name="combined_a")
        combined_b = concatenate([max_pool_b, avg_pool_b], name="combined_b")
        
        # Dense projection
        dense_layer = Dense(256, activation='relu', name="shared_dense")
        bn_layer = BatchNormalization(name="shared_bn")
        dropout_layer = Dropout(0.4, name="shared_dropout")
        
        processed_a = dropout_layer(bn_layer(dense_layer(combined_a)))
        processed_b = dropout_layer(bn_layer(dense_layer(combined_b)))
        
        # Final dense for embeddings
        embedding_dense = Dense(128, activation='relu', name="embedding_dense")
        
        embedding_a = embedding_dense(processed_a)
        embedding_b = embedding_dense(processed_b)
        
        # Normalize embeddings
        norm_a = Lambda(lambda x: K.l2_normalize(x, axis=1), name="norm_a")(embedding_a)
        norm_b = Lambda(lambda x: K.l2_normalize(x, axis=1), name="norm_b")(embedding_b)
        
        # Multiple similarity measures
        # 1. L1 distance (Manhattan)
        l1_distance = Lambda(
            lambda x: K.abs(x[0] - x[1]),
            name="l1_distance"
        )([norm_a, norm_b])
        
        # 2. L2 distance (Euclidean)
        l2_distance = Lambda(
            lambda x: K.sqrt(K.sum(K.square(x[0] - x[1]), axis=-1, keepdims=True) + K.epsilon()),
            name="l2_distance"
        )([norm_a, norm_b])
        
        # 3. Cosine similarity
        cosine = Lambda(
            lambda x: K.sum(x[0] * x[1], axis=-1, keepdims=True),
            name="cosine_similarity"
        )([norm_a, norm_b])
        
        # 4. Element-wise product
        product = Multiply(name="element_product")([norm_a, norm_b])
        
        # 5. Squared difference
        squared_diff = Lambda(
            lambda x: K.square(x[0] - x[1]),
            name="squared_diff"
        )([norm_a, norm_b])
        
        # Combine all features
        combined = concatenate([
            l1_distance,
            l2_distance,
            cosine,
            product,
            squared_diff
        ], name="combined_features")
        
        # Classification head
        dense1 = Dense(256, activation='relu', name="classifier_dense1")(combined)
        dense1 = BatchNormalization(name="classifier_bn1")(dense1)
        dense1 = Dropout(0.5, name="classifier_dropout1")(dense1)
        
        dense2 = Dense(128, activation='relu', name="classifier_dense2")(dense1)
        dense2 = BatchNormalization(name="classifier_bn2")(dense2)
        dense2 = Dropout(0.4, name="classifier_dropout2")(dense2)
        
        dense3 = Dense(64, activation='relu', name="classifier_dense3")(dense2)
        dense3 = Dropout(0.3, name="classifier_dropout3")(dense3)
        
        # Output
        output = Dense(1, activation='sigmoid', name="similarity_score")(dense3)
        
        model = Model(
            inputs=[input_a, input_b],
            outputs=output,
            name=f"Hybrid_Siamese_{rnn_type.upper()}"
        )
        
        return model
    
    # ==================== MODEL 3: ATTENTION-BASED SIAMESE ====================
    def create_attention_siamese(self, rnn_type='lstm'):
        """
        Siamese with attention mechanism
        Helps focus on important characters
        """
        input_a = Input(shape=(self.max_len,), name="input_a")
        input_b = Input(shape=(self.max_len,), name="input_b")
        
        # Embedding
        embedding_layer = Embedding(
            input_dim=self.vocab_size + 1,
            output_dim=self.embed_dim,
            input_length=self.max_len,
            mask_zero=True,
            embeddings_initializer='glorot_uniform',
            name="shared_embedding"
        )
        
        embedded_a = embedding_layer(input_a)
        embedded_b = embedding_layer(input_b)
        
        # RNN cell
        if rnn_type.lower() == 'lstm':
            rnn_cell = LSTM
        elif rnn_type.lower() == 'gru':
            rnn_cell = GRU
        else:
            rnn_cell = SimpleRNN
        
        # Bidirectional RNN with sequences
        bi_rnn = Bidirectional(
            rnn_cell(self.rnn_units, return_sequences=True, dropout=0.3),
            name="shared_bi_rnn"
        )
        
        rnn_a = bi_rnn(embedded_a)
        rnn_b = bi_rnn(embedded_b)
        
        # Self-attention mechanism
        attention_layer_a = tf.keras.layers.MultiHeadAttention(
            num_heads=4,
            key_dim=self.rnn_units // 2,
            name="attention_a"
        )
        
        attention_layer_b = tf.keras.layers.MultiHeadAttention(
            num_heads=4,
            key_dim=self.rnn_units // 2,
            name="attention_b"
        )
        
        attended_a = attention_layer_a(rnn_a, rnn_a)
        attended_b = attention_layer_b(rnn_b, rnn_b)
        
        # Pooling
        pooled_a = GlobalMaxPooling1D(name="pool_a")(attended_a)
        pooled_b = GlobalMaxPooling1D(name="pool_b")(attended_b)
        
        # Normalize
        norm_a = Lambda(lambda x: K.l2_normalize(x, axis=1), name="norm_a")(pooled_a)
        norm_b = Lambda(lambda x: K.l2_normalize(x, axis=1), name="norm_b")(pooled_b)
        
        # Multiple similarities
        l1 = Lambda(lambda x: K.abs(x[0] - x[1]), name="l1")([norm_a, norm_b])
        cosine = Lambda(lambda x: K.sum(x[0] * x[1], axis=-1, keepdims=True), name="cosine")([norm_a, norm_b])
        product = Multiply(name="product")([norm_a, norm_b])
        
        combined = concatenate([l1, cosine, product], name="combined")
        
        # Dense layers
        dense1 = Dense(128, activation='relu', name="dense1")(combined)
        dense1 = BatchNormalization(name="bn1")(dense1)
        dense1 = Dropout(0.4, name="dropout1")(dense1)
        
        dense2 = Dense(64, activation='relu', name="dense2")(dense1)
        dense2 = Dropout(0.3, name="dropout2")(dense2)
        
        output = Dense(1, activation='sigmoid', name="output")(dense2)
        
        model = Model(
            inputs=[input_a, input_b],
            outputs=output,
            name=f"Attention_Siamese_{rnn_type.upper()}"
        )
        
        return model
    
    # ==================== COMPILATION ====================
    def compile_contrastive_model(self, model, learning_rate=0.001):
        """Compile model with contrastive loss"""
        optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
        
        model.compile(
            optimizer=optimizer,
            loss=lambda y_true, y_pred: self.contrastive_loss(y_true, y_pred, margin=self.margin),
            metrics=[
                tf.keras.metrics.MeanAbsoluteError(name='mae'),
            ]
        )
        
        return model
    
    def compile_hybrid_model(self, model, learning_rate=0.001):
        """Compile hybrid model with binary crossentropy"""
        optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
        
        model.compile(
            optimizer=optimizer,
            loss='binary_crossentropy',
            metrics=[
                'accuracy',
                tf.keras.metrics.Precision(name='precision'),
                tf.keras.metrics.Recall(name='recall'),
                tf.keras.metrics.AUC(name='auc'),
                tf.keras.metrics.BinaryAccuracy(name='binary_accuracy')
            ]
        )
        
        return model
    
    def model_summary(self, model):
        """Print model summary"""
        print("=" * 80)
        print(f"Model: {model.name}")
        print("=" * 80)
        model.summary()
        
        trainable_params = np.sum([K.count_params(w) for w in model.trainable_weights])
        non_trainable_params = np.sum([K.count_params(w) for w in model.non_trainable_weights])
        
        print(f"\nTotal parameters: {trainable_params + non_trainable_params:,}")
        print(f"Trainable parameters: {trainable_params:,}")
        print(f"Non-trainable parameters: {non_trainable_params:,}")
        print("=" * 80)

In [26]:
# ==================== STEP 1: INITIALIZE IMPROVED MODEL MANAGER ====================
print("="*80)
print("INITIALIZING IMPROVED SIAMESE MODELS WITH CONTRASTIVE LEARNING")
print("="*80)

VOCAB_SIZE = len(char2idx)
MAX_LEN = max_len

# Use the NEW improved model class
model_manager = ImprovedSiameseKhmerSimilarity(
    vocab_size=VOCAB_SIZE,
    max_len=MAX_LEN,
    embed_dim=128,  # Increased from 64
    rnn_units=128   # Increased from 64
)

# ==================== STEP 2: CREATE IMPROVED MODELS ====================
print("\n" + "="*80)
print("Creating Hybrid LSTM (BEST - Uses multiple similarity measures)...")
print("="*80)
hybrid_lstm_model = model_manager.create_hybrid_siamese(rnn_type='lstm')
hybrid_lstm_model = model_manager.compile_hybrid_model(hybrid_lstm_model, learning_rate=0.001)
model_manager.model_summary(hybrid_lstm_model)

print("\n" + "="*80)
print("Creating Attention LSTM (Uses multi-head attention)...")
print("="*80)
attention_lstm_model = model_manager.create_attention_siamese(rnn_type='lstm')
attention_lstm_model = model_manager.compile_hybrid_model(attention_lstm_model, learning_rate=0.001)
model_manager.model_summary(attention_lstm_model)

print("\n" + "="*80)
print("Creating Contrastive LSTM (Pure distance-based learning)...")
print("="*80)
contrastive_lstm_model = model_manager.create_contrastive_siamese(rnn_type='lstm')
contrastive_lstm_model = model_manager.compile_contrastive_model(contrastive_lstm_model, learning_rate=0.001)
model_manager.model_summary(contrastive_lstm_model)

print("\n" + "="*80)
print("Creating Hybrid GRU (GRU variant)...")
print("="*80)
hybrid_gru_model = model_manager.create_hybrid_siamese(rnn_type='gru')
hybrid_gru_model = model_manager.compile_hybrid_model(hybrid_gru_model, learning_rate=0.001)
model_manager.model_summary(hybrid_gru_model)

print("\n" + "="*80)
print("Creating Attention GRU (GRU with attention)...")
print("="*80)
attention_gru_model = model_manager.create_attention_siamese(rnn_type='gru')
attention_gru_model = model_manager.compile_hybrid_model(attention_gru_model, learning_rate=0.001)
model_manager.model_summary(attention_gru_model)

print("\n" + "="*100)
print("ALL MODELS CREATED SUCCESSFULLY!")
print("="*100)

# ==================== STEP 3: DATA SPLIT ====================
import tensorflow as tf
from sklearn.model_selection import train_test_split
import os
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pickle

print("\n" + "="*80)
print("SPLITTING DATA INTO TRAIN/TEST SETS")
print("="*80)

X1_train, X1_test, X2_train, X2_test, y_train, y_test = train_test_split(
    X1, X2, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # Keep class balance
)

print(f"Training samples:   {len(y_train):,}")
print(f"Test samples:       {len(y_test):,}")
print(f"Positive ratio (train): {y_train.mean():.4f}")
print(f"Positive ratio (test):  {y_test.mean():.4f}")

# ==================== SAVE PREPROCESSING DATA ====================
print("\n" + "="*80)
print("SAVING PREPROCESSING DATA (char2idx, max_len)")
print("="*80)

try:
    with open('char2idx.pkl', 'wb') as f:
        pickle.dump(char2idx, f)
    print(f"✓ Saved char2idx.pkl ({len(char2idx)} characters)")
    
    with open('max_len.pkl', 'wb') as f:
        pickle.dump(max_len, f)
    print(f"✓ Saved max_len.pkl (max_len={max_len})")
except Exception as e:
    print(f"⚠ Warning: Could not save preprocessing data: {e}")

# ==================== STEP 4: TRAINING FUNCTION ====================
def train_improved_model(model, model_name, X1_train, X2_train, y_train, 
                        X1_test, X2_test, y_test, epochs=100, batch_size=64):
    """
    Train improved model with better settings
    """
    print(f"\n{'='*80}")
    print(f"TRAINING: {model_name.upper()}")
    print(f"{'='*80}")
    
    # 🔹 ADDED: Print model type info
    if "contrastive" in model_name.lower():
        print("📊 Model Type: CONTRASTIVE LOSS (Distance-based)")
        print("   → Output: Distance between embeddings")
        print("   → Similar pairs: Low distance")
        print("   → Different pairs: High distance")
    else:
        print("📊 Model Type: BINARY CLASSIFICATION")
        print("   → Output: Similarity probability (0-1)")
        print("   → Similar pairs: Score close to 1")
        print("   → Different pairs: Score close to 0")
    print()
    
    os.makedirs("saved_models", exist_ok=True)
    
    # Determine which metrics to monitor based on model type
    has_auc = 'auc' in [m.name for m in model.metrics]
    monitor_metric = 'val_auc' if has_auc else 'val_loss'
    monitor_mode = 'max' if has_auc else 'min'
    
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor=monitor_metric,
            mode=monitor_mode,
            patience=15,  # More patience
            restore_best_weights=True,
            verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor=monitor_metric,
            mode=monitor_mode,
            factor=0.5,
            patience=7,
            min_lr=1e-7,
            verbose=1
        ),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=f"saved_models/best_{model_name}.h5",
            monitor=monitor_metric,
            mode=monitor_mode,
            save_best_only=True,
            verbose=1
        )
    ]
    
    # Train
    history = model.fit(
        [X1_train, X2_train], y_train,
        validation_data=([X1_test, X2_test], y_test),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=1
    )
    
    # Final evaluation
    print(f"\n{'='*80}")
    print(f"FINAL EVALUATION - {model_name.upper()}")
    print(f"{'='*80}")
    
    eval_results = model.evaluate([X1_test, X2_test], y_test, verbose=0)
    
    # Print all metrics
    for metric_name, metric_value in zip(model.metrics_names, eval_results):
        print(f"{metric_name:20s}: {metric_value:.4f}")
    
    # Calculate F1 if precision and recall are available
    if 'precision' in model.metrics_names and 'recall' in model.metrics_names:
        prec_idx = model.metrics_names.index('precision')
        rec_idx = model.metrics_names.index('recall')
        prec = eval_results[prec_idx]
        rec = eval_results[rec_idx]
        f1 = 2 * (prec * rec) / (prec + rec + 1e-8)
        print(f"{'f1_score':20s}: {f1:.4f}")
    
    # Get best validation metric
    if has_auc:
        best_metric = max(history.history['val_auc'])
        print(f"{'Best Val AUC':20s}: {best_metric:.4f}")
    else:
        best_metric = min(history.history['val_loss'])
        print(f"{'Best Val Loss':20s}: {best_metric:.4f}")
    
    # Save final model
    model.save(f"saved_models/final_{model_name}.h5")
    print(f"\n✓ Saved: saved_models/best_{model_name}.h5")
    print(f"✓ Saved: saved_models/final_{model_name}.h5")
    
    return history, model

# ==================== STEP 5: TRAIN ALL MODELS ====================
print("\n" + "="*100)
print("STARTING TRAINING - THIS WILL TAKE A WHILE!")
print("="*100)

# 🌟 Train Hybrid LSTM (RECOMMENDED - Best overall)
print("\n🌟 TRAINING MODEL 1/5: HYBRID LSTM")
history_hybrid_lstm, trained_hybrid_lstm = train_improved_model(
    model=hybrid_lstm_model,
    model_name="hybrid_lstm",
    X1_train=X1_train, X2_train=X2_train, y_train=y_train,
    X1_test=X1_test, X2_test=X2_test, y_test=y_test,
    epochs=100,
    batch_size=64
)

# 👁️ Train Attention LSTM
print("\n👁️ TRAINING MODEL 2/5: ATTENTION LSTM")
history_att_lstm, trained_att_lstm = train_improved_model(
    model=attention_lstm_model,
    model_name="attention_lstm",
    X1_train=X1_train, X2_train=X2_train, y_train=y_train,
    X1_test=X1_test, X2_test=X2_test, y_test=y_test,
    epochs=100,
    batch_size=64
)

# 📏 Train Contrastive LSTM (Uses contrastive loss!)
print("\n📏 TRAINING MODEL 3/5: CONTRASTIVE LSTM")
history_cont_lstm, trained_cont_lstm = train_improved_model(
    model=contrastive_lstm_model,
    model_name="contrastive_lstm",
    X1_train=X1_train, X2_train=X2_train, y_train=y_train,
    X1_test=X1_test, X2_test=X2_test, y_test=y_test,
    epochs=100,
    batch_size=64
)

# 🔄 Train Hybrid GRU
print("\n🔄 TRAINING MODEL 4/5: HYBRID GRU")
history_hybrid_gru, trained_hybrid_gru = train_improved_model(
    model=hybrid_gru_model,
    model_name="hybrid_gru",
    X1_train=X1_train, X2_train=X2_train, y_train=y_train,
    X1_test=X1_test, X2_test=X2_test, y_test=y_test,
    epochs=100,
    batch_size=64
)

# 👁️🔄 Train Attention GRU
print("\n👁️🔄 TRAINING MODEL 5/5: ATTENTION GRU")
history_att_gru, trained_att_gru = train_improved_model(
    model=attention_gru_model,
    model_name="attention_gru",
    X1_train=X1_train, X2_train=X2_train, y_train=y_train,
    X1_test=X1_test, X2_test=X2_test, y_test=y_test,
    epochs=100,
    batch_size=64
)

# ==================== STEP 6: TESTING UTILITIES ====================
def predict_similarity(model, word1, word2, char2idx, max_len):
    """Predict similarity between two Khmer words"""
    def encode(chars):
        return [char2idx.get(c, 0) for c in chars]
    
    seq1 = pad_sequences(
        [encode(list(word1))],
        maxlen=max_len,
        padding="post",
        truncating="post"
    )
    seq2 = pad_sequences(
        [encode(list(word2))],
        maxlen=max_len,
        padding="post",
        truncating="post"
    )
    
    score = model.predict([seq1, seq2], verbose=0)[0][0]
    
    # For contrastive models, convert distance to similarity
    if "contrastive" in model.name.lower():
        score = 1.0 / (1.0 + score)  # Convert distance to similarity
    
    return float(score)

def is_similar(model, word1, word2, char2idx, max_len, threshold=0.9):
    """Check if words are similar (threshold: 0.9)"""
    score = predict_similarity(model, word1, word2, char2idx, max_len)
    return score, score >= threshold

def test_model_on_examples(model, model_name, test_pairs, char2idx, max_len):
    """Test a model on example word pairs"""
    print(f"\n{'='*80}")
    print(f"TESTING MODEL: {model_name.upper()}")
    print(f"{'='*80}")
    
    for w1, w2 in test_pairs:
        score, decision = is_similar(model, w1, w2, char2idx, max_len)
        status = "✓ Similar" if decision else "✗ Not similar"
        print(f"{w1:15s} vs {w2:15s}: {score:.4f} → {status}")
    print()

# ==================== STEP 7: TEST ALL MODELS ====================
print("\n" + "="*100)
print("TESTING ALL MODELS ON EXAMPLE PAIRS")
print("="*100)

# Test cases
test_pairs = [
    ("អាយុ", "ជន្ម"),      # Should be similar (age/birth)
    ("ទី", "ស្រីកល្យាណី"),  # Should NOT be similar
    ("ច័ន្ទ", "ខែ"),        # Should be similar (moon/month)
    ("ទឹក", "អាង"),         # Should NOT be similar (water vs pond)
]

# Test all trained models
test_model_on_examples(trained_hybrid_lstm, "Hybrid LSTM", test_pairs, char2idx, max_len)
test_model_on_examples(trained_att_lstm, "Attention LSTM", test_pairs, char2idx, max_len)
test_model_on_examples(trained_cont_lstm, "Contrastive LSTM 📏", test_pairs, char2idx, max_len)
test_model_on_examples(trained_hybrid_gru, "Hybrid GRU", test_pairs, char2idx, max_len)
test_model_on_examples(trained_att_gru, "Attention GRU", test_pairs, char2idx, max_len)

# ==================== STEP 8: SUMMARY ====================
print("\n" + "="*100)
print("TRAINING AND TESTING COMPLETED!")
print("="*100)
print("\n📦 SAVED MODELS (Improved with Contrastive Learning):")
print("\n🌟 HYBRID MODELS (Binary Classification - Recommended):")
print("  • saved_models/best_hybrid_lstm.h5       ⭐ BEST OVERALL")
print("  • saved_models/final_hybrid_lstm.h5")
print("  • saved_models/best_hybrid_gru.h5")
print("  • saved_models/final_hybrid_gru.h5")
print("\n👁️ ATTENTION MODELS (With Multi-Head Attention):")
print("  • saved_models/best_attention_lstm.h5")
print("  • saved_models/final_attention_lstm.h5")
print("  • saved_models/best_attention_gru.h5")
print("  • saved_models/final_attention_gru.h5")
print("\n📏 CONTRASTIVE MODEL (Distance-based with Contrastive Loss):")
print("  • saved_models/best_contrastive_lstm.h5  🎯 Uses Contrastive Loss!")
print("  • saved_models/final_contrastive_lstm.h5")
print("\n💾 PREPROCESSING DATA:")
print("  • char2idx.pkl")
print("  • max_len.pkl")
print("\n🎯 RECOMMENDATION:")
print("  Primary: best_hybrid_lstm.h5 (Best accuracy & reliability)")
print("  Alternative: best_contrastive_lstm.h5 (Pure distance learning)")
print("\n📊 EXPECTED PERFORMANCE: 0.90+ AUC, accurate similarity detection")
print("="*100)

# ==================== STEP 9: MODEL TYPE EXPLANATION ====================
print("\n" + "="*100)
print("📚 UNDERSTANDING YOUR MODELS")
print("="*100)
print("""
🌟 HYBRID MODELS (hybrid_lstm, hybrid_gru):
   • Loss Function: Binary Cross-Entropy
   • Output: Probability (0-1)
   • Uses: L1, L2, Cosine, Product, Squared Difference
   • Best for: General similarity detection
   • Similar pairs: Score close to 1.0
   • Different pairs: Score close to 0.0

👁️ ATTENTION MODELS (attention_lstm, attention_gru):
   • Loss Function: Binary Cross-Entropy
   • Output: Probability (0-1)
   • Uses: Multi-head attention + similarity measures
   • Best for: Complex patterns, long words
   • Similar pairs: Score close to 1.0
   • Different pairs: Score close to 0.0

📏 CONTRASTIVE MODEL (contrastive_lstm):
   • Loss Function: CONTRASTIVE LOSS 🎯
   • Output: Distance (converted to similarity in testing)
   • Formula: Loss = y*d² + (1-y)*max(margin-d, 0)²
   • Best for: Learning semantic embeddings
   • Similar pairs: Low distance (high similarity score)
   • Different pairs: High distance (low similarity score)
   • This is the "contrastive learning" your teacher mentioned!
""")

# ==================== STEP 10: HOW TO LOAD AND USE ====================
print("\n" + "="*100)
print("HOW TO LOAD AND USE SAVED MODELS")
print("="*100)
print("""
# Load the best model:
from tensorflow.keras.models import load_model
import pickle
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load preprocessing data
with open('char2idx.pkl', 'rb') as f:
    char2idx = pickle.load(f)
with open('max_len.pkl', 'rb') as f:
    max_len = pickle.load(f)

# Load model
model = load_model('saved_models/best_hybrid_lstm.h5')

# Make predictions:
score = predict_similarity(model, "អាយុ", "ជន្ម", char2idx, max_len)
print(f"Similarity score: {score:.4f}")

# Check if similar (threshold 0.9):
score, is_sim = is_similar(model, "អាយុ", "ជន្ម", char2idx, max_len, threshold=0.9)
print(f"Similar: {is_sim}")

# ===== TO USE CONTRASTIVE MODEL =====
# The contrastive model outputs distance, but predict_similarity()
# automatically converts it to similarity score, so you use it the same way!
model_contrastive = load_model('saved_models/best_contrastive_lstm.h5')
score = predict_similarity(model_contrastive, "អាយុ", "ជន្ម", char2idx, max_len)
print(f"Contrastive similarity score: {score:.4f}")
""")

INITIALIZING IMPROVED SIAMESE MODELS WITH CONTRASTIVE LEARNING

Creating Hybrid LSTM (BEST - Uses multiple similarity measures)...
Model: Hybrid_Siamese_LSTM


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:965: UserWarning: Layer 'max_pool_a' (of type GlobalMaxPooling1D) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:965: UserWarning: Layer 'max_pool_b' (of type GlobalMaxPooling1D) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Model: "Hybrid_Siamese_LSTM"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_a             │ (None, 100)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_b             │ (None, 100)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_embedding    │ (None, 100, 128)  │      9,984 │ input_a[0][0],    │
│ (Embedding)         │                   │            │ input_b[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_10        │ (None, 100)       │          0 │ input_a[0][0]     │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_11        │ (None, 100)       │          0 │ input_b[0][0]     │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_bi_rnn       │ (None, 100, 256)  │    263,168 │ shared_embedding… │
│ (Bidirectional)     │                   │            │ not_equal_10[0][… │
│                     │                   │            │ shared_embedding… │
│                     │                   │            │ not_equal_11[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pool_a          │ (None, 256)       │          0 │ shared_bi_rnn[0]… │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ avg_pool_a          │ (None, 256)       │          0 │ shared_bi_rnn[0]… │
│ (GlobalAveragePool… │                   │            │ not_equal_10[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pool_b          │ (None, 256)       │          0 │ shared_bi_rnn[1]… │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ avg_pool_b          │ (None, 256)       │          0 │ shared_bi_rnn[1]… │
│ (GlobalAveragePool… │                   │            │ not_equal_11[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ combined_a          │ (None, 512)       │          0 │ max_pool_a[0][0], │
│ (Concatenate)       │                   │            │ avg_pool_a[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ combined_b          │ (None, 512)       │          0 │ max_pool_b[0][0], │
│ (Concatenate)       │                   │            │ avg_pool_b[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_dense        │ (None, 256)       │    131,328 │ combined_a[0][0], │
│ (Dense)             │                   │            │ combined_b[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_bn           │ (None, 256)       │      1,024 │ shared_dense[0][… │
│ (BatchNormalizatio… │                   │            │ shared_dense[1][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_dropout      │ (None, 256)       │          0 │ shared_bn[0][0],  │
│ (Dropout)           │                   │            │ shared_bn[1][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_dense     │ (None, 128)       │     32,896 │ shared_dropout[0… │
│ (Dense)             │                   │            │ shared_dropout[1

 Total params: 580,225 (2.21 MB)

 Trainable params: 578,945 (2.21 MB)

... (145 outputs left out)

150/150 ━━━━━━━━━━━━━━━━━━━━ 6s 43ms/step - accuracy: 0.9071 - auc: 0.9450 - binary_accuracy: 0.9071 - loss: 0.2483 - precision: 0.8583 - recall: 0.7494 - val_accuracy: 0.8721 - val_auc: 0.9073 - val_binary_accuracy: 0.8721 - val_loss: 0.3207 - val_precision: 0.7676 - val_recall: 0.7217 - learning_rate: 0.0010
Epoch 28/100
149/150 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.8968 - auc: 0.9432 - binary_accuracy: 0.8968 - loss: 0.2539 - precision: 0.8412 - recall: 0.7331
Epoch 28: val_loss did not improve from 0.32069
150/150 ━━━━━━━━━━━━━━━━━━━━ 6s 42ms/step - accuracy: 0.8968 - auc: 0.9432 - binary_accuracy: 0.8968 - loss: 0.2539 - precision: 0.8412 - recall: 0.7332 - val_accuracy: 0.8338 - val_auc: 0.9047 - val_binary_accuracy: 0.8338 - val_loss: 0.3900 - val_precision: 0.6417 - val_recall: 0.8026 - learning_rate: 0.0010
Epoch 29/100
149/150 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.9023 - auc: 0.9487 - binary_accuracy: 0.9023 - loss: 0.2419 - precision: 0.8549 - recall: 0.7

150/150 ━━━━━━━━━━━━━━━━━━━━ 6s 43ms/step - accuracy: 0.9071 - auc: 0.9566 - binary_accuracy: 0.9071 - loss: 0.2259 - precision: 0.8583 - recall: 0.7661 - val_accuracy: 0.8754 - val_auc: 0.9232 - val_binary_accuracy: 0.8754 - val_loss: 0.3001 - val_precision: 0.8158 - val_recall: 0.6667 - learning_rate: 0.0010
Epoch 32/100
149/150 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.9103 - auc: 0.9546 - binary_accuracy: 0.9103 - loss: 0.2295 - precision: 0.8634 - recall: 0.7748
Epoch 32: val_loss did not improve from 0.30012
150/150 ━━━━━━━━━━━━━━━━━━━━ 6s 43ms/step - accuracy: 0.9102 - auc: 0.9545 - binary_accuracy: 0.9102 - loss: 0.2296 - precision: 0.8633 - recall: 0.7747 - val_accuracy: 0.8704 - val_auc: 0.9181 - val_binary_accuracy: 0.8704 - val_loss: 0.3227 - val_precision: 0.7402 - val_recall: 0.7654 - learning_rate: 0.0010
Epoch 33/100
149/150 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.9120 - auc: 0.9598 - binary_accuracy: 0.9120 - loss: 0.2167 - precision: 0.8628 - recall: 0.7

150/150 ━━━━━━━━━━━━━━━━━━━━ 6s 42ms/step - accuracy: 0.9194 - auc: 0.9680 - binary_accuracy: 0.9194 - loss: 0.1960 - precision: 0.8711 - recall: 0.8047 - val_accuracy: 0.8863 - val_auc: 0.9304 - val_binary_accuracy: 0.8863 - val_loss: 0.2961 - val_precision: 0.8632 - val_recall: 0.6634 - learning_rate: 0.0010
Epoch 37/100
149/150 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.9224 - auc: 0.9666 - binary_accuracy: 0.9224 - loss: 0.1957 - precision: 0.8796 - recall: 0.8098
Epoch 37: val_loss did not improve from 0.29613
150/150 ━━━━━━━━━━━━━━━━━━━━ 6s 42ms/step - accuracy: 0.9223 - auc: 0.9666 - binary_accuracy: 0.9223 - loss: 0.1959 - precision: 0.8795 - recall: 0.8096 - val_accuracy: 0.8904 - val_auc: 0.9226 - val_binary_accuracy: 0.8904 - val_loss: 0.3190 - val_precision: 0.8660 - val_recall: 0.6796 - learning_rate: 0.0010
Epoch 38/100
149/150 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.9164 - auc: 0.9658 - binary_accuracy: 0.9164 - loss: 0.2009 - precision: 0.8641 - recall: 0.7

loss                : 0.2961
compile_metrics     : 0.8863
Best Val Loss       : 0.2961

✓ Saved: saved_models/best_attention_gru.h5
✓ Saved: saved_models/final_attention_gru.h5

TESTING ALL MODELS ON EXAMPLE PAIRS

TESTING MODEL: HYBRID LSTM
អាយុ            vs ជន្ម           : 0.9569 → ✓ Similar
ទី              vs ស្រីកល្យាណី    : 0.3201 → ✗ Not similar
ច័ន្ទ           vs ខែ             : 0.9736 → ✓ Similar
ទឹក             vs អាង            : 0.4450 → ✗ Not similar


TESTING MODEL: ATTENTION LSTM
អាយុ            vs ជន្ម           : 0.9807 → ✓ Similar
ទី              vs ស្រីកល្យាណី    : 0.0134 → ✗ Not similar
ច័ន្ទ           vs ខែ             : 0.9614 → ✓ Similar
ទឹក             vs អាង            : 0.0001 → ✗ Not similar


TESTING MODEL: CONTRASTIVE LSTM 📏
អាយុ            vs ជន្ម           : 0.4707 → ✗ Not similar
ទី              vs ស្រីកល្យាណី    : 0.4026 → ✗ Not similar
ច័ន្ទ           vs ខែ             : 0.4766 → ✗ Not similar
ទឹក             vs អាង            : 0.5243 → ✗ Not simila